# FahMaiRAG สรุปสิ่งที่ทำใน hackathon ครั้งที่ 3

ระบบนี้คือ RAG ที่ออกแบบมาเพื่อตอบคำถามเกี่ยวกับสินค้าไอที นโยบายการรับประกัน และการคำนวณสิทธิประโยชน์สมาชิกของ "ร้านฟ้าใหม่"
ใช้เทคนิคดั่งนี้ครับ โดยจะเป็นเป็นเทคนิคเดียวกันทั้งหมด แต่สลับโมเดล, Top k,chunk_sizem chunk_overlap เพื่อลองจูนความแม่นยำเอาครับ
## Document Processing
- Metadata Extraction: ใช้ Regular Expression ดึงข้อมูลสำคัญ เช่น แบรนด์, หมวดหมู่, รหัสสินค้า และราคาขั้นต่ำ ออกมาจากไฟล์ Markdown อัตโนมัติ เพื่อใช้เป็นตัวกรอง ในภายหลัง
คิดว่าน่าจะช่วยให้ค้นหาข้อมูลได้แม่นยำมากขึ้นครับ
- Context Chunking: หั่นเอกสารด้วย MarkdownHeaderTextSplitter ควบคู่กับ RecursiveCharacterTextSplitter และมีการแทรก Context Header เข้าไปในทุกๆ Chunk เพื่อให้ LLM ไม่หลงบริบทว่ากำลังอ่านข้อมูลของสินค้าตัวไหนอยู่

- MarkdownHeaderTextSplitter และRecursiveCharacterTextSplitter ผมได้ไปนั่งดูในไฟล์เอกสารมาแบ่งความสำคัญได้ดั่งนี้ครับ `"#":"H1_Title"`,`"##":"H2_Section"`, `"###":"H3_Subsection"` และ `"\n\n"`, `"\n"`,  `" "`, `""` ตามลำดับครับ

- **ทำไมถึงทำ** ผมค้นพบว่าในเอกสารมี ---, ** จำนวนมาก บางหน้ามี บางหน้าไม่มี จึงทำการ ทำให้ทุก chunk มี metadata ไว้บนสุดเป็น header มันจะช่วยให้ค้นหาได้แม่นยำขึ้นครับและ เอกสารที่ให้มา ค่อนข้างแบ่งเป็นระเบียบเรียบร้อยด้วย markdown โดยผมทำการลบ ---, **, (,) ออกเพื่อความสะอาดของข้อมูล จะได้เป็น format เดี่ยวกันทั้งหมดคือไม่มี --- ขั้น และไม่เน้นตัวหนาให้คนอ่านง่าย และ (,) เพื่อจำนวนตัวเลข เช่น ราคา price, ความจุ mah ไม่ให้ LLM อ่านแล้วงง ว่าสรุปคือตัวเลข หรือ สองคำที่คั่นด้วย (,) กันแน่

## Hybrid Retrieval + Reranking

Query Expansion: ส่งคำถามให้ LLM สกัด "Keyword" ที่แท้จริงออกมาก่อน เพื่อแก้ปัญหาผู้ใช้พิมพ์คำถามยาวหรือใช้คำสร้อย

Ensemble Search: ผสานพลังการค้นหา 2 รูปแบบ:

- Vector Search ใช้ FAISS + BGE-M3 ค้นหาด้วยความหมายเชิงบริบท (Semantic)

- Keyword Search ใช้ BM25 ค้นหาด้วยการเปรียบเทียบคำเป๊ะๆ ป้องกันการตกหล่นของรหัสสินค้าหรือชื่อรุ่น

- Cross-Encoder Reranking: นำผลลัพธ์ที่ได้จากทั้งสองระบบมารวมกัน แล้วให้โมเดล BGE-Reranker-v2-m3 ให้คะแนนความเกี่ยวข้องใหม่ เพื่อคัดเฉพาะ Chunk ที่แม่นยำที่สุด 5 อันดับแรกส่งให้ LLM

## Agentic Routing & Hard Filtering
- Budget Extraction: มีระบบสกัด "งบประมาณ" จากคำถาม (เช่น "ราคาไม่เกิน 20,000") เพื่อนำมาตัดสินค้าที่ราคาเกินงบทิ้งไปตั้งแต่ชั้น Retrieval

- Policy Intent Detection: ดักจับคำสำคัญ (เช่น เคลม, แตก, พัง, จัดส่ง) หากพบว่าเป็นคำถามแนวนโยบาย ระบบจะบังคับดึงไฟล์ FAQ/Policy มาให้ LLM อ่านเสมอ เพื่อไม่ให้ตอบผิดกฎของร้าน

## Chain-of-Thought Prompt
- ในขั้นตอนการตอบ มีการวางกฎเหล็กเรื่องการคำนวณพอยต์และค่าจัดส่งไว้อย่างชัดเจน

- บังคับให้ LLM แสดงวิธีทำในแท็ก <reasoning> ... </reasoning> ก่อนที่จะสรุปคำตอบสุดท้ายเป็นตัวเลขในแท็ก <answer> ... </answer> เพื่อลดอาการ Hallucination หรือมันเอ๋อนั้นแหละ

- มีการส่งรายชื่อของแบรนไปให้ LLM ตัวที่สกัด key word ดูเพื่อที่จะให้มันเรียนรู้จะได้ไม่เอาชื่อแบรนไปปนกับอย่างอื่น ลดความเวิ่นเว่อของ LLM

- ปรับ temperature ให้เป็น 0.0 เพื่อบอกมันว่าให้เดาคำหรือให้เพ้อเจ้อน้อยที่สุด

## Retry request Pipeline
- มีลูป try-except แบบ Infinite Retry หาก API ของ LLM เกิดอาการ 502 Bad Gateway หรือล่ม ระบบจะรอ 3 วินาทีแล้วยิงคำถามซ้ำอัตโนมัติจนกว่าจะสำเร็จครบ 100 ข้อ พร้อมบันทึก Log เอามาใช้วิเคราะห์ที่AIมันคิดและตอบด้วย ว่าควรพัมนาไปทางไหนต่อ

---

### สุดท้ายนี้ขอขอบคุณทีมงานทุกท่านที่ทำงานกันอย่างหนักตลอด 1เดือนที่ผ่านมา โครงการดีจริงๆทำให้หลายๆคนพัฒนาไปเยอะมากๆเลย รวมถึงผมด้วย :))

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface

!pip install -U sentence-transformers rank_bm25 openai pandas faiss-cpu

In [ ]:
!unzip /content/super-ai-engineer-s-6-fah-mai-rag-challenge-level-1.zip

In [8]:
import os
import re
import pandas as pd
import requests
from typing import List
import time

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

from sentence_transformers import CrossEncoder

In [14]:
def load_and_process_docs(data_dir: str) -> List[Document]:
    print("กำลังสกัด Metadata และสร้าง Chunks...")

    valid_brands = set()
    valid_categories = set()
    valid_skus = set()

    headers_to_split_on = [
        ("#", "H1_Title"),
        ("##", "H2_Section"),
        ("###", "H3_Subsection")
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2048,
        chunk_overlap=300,
        separators=["\n\n", "\n", " ", ""]
    )

    processed_docs = []
    file_count = 0

    for root, dirs, files in os.walk(data_dir):
        for filename in files:
            if filename.endswith(".md"):
                file_path = os.path.join(root, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        file_count += 1
                        text = f.read()

                        clean_text = text.replace('**', '')
                        clean_text = clean_text.replace('---', '')

                        brand = re.search(r"แบรนด์:\s*(.*)", clean_text)
                        sku = re.search(r"รหัสสินค้า:\s*(.*)", clean_text)
                        category = re.search(r"หมวดหมู่:\s*(.*)", clean_text)

                        prices_found = re.findall(r"ราคา:\s*฿?([\d,]+)", clean_text)
                        price_str = ", ".join(prices_found) if prices_found else "N/A"

                        price_ints = [int(p.replace(',', '')) for p in prices_found if p.replace(',', '').isdigit()]
                        min_price = min(price_ints) if price_ints else 0

                        b_val = brand.group(1).split('—')[0].strip() if brand else "ทั่วไป"
                        s_val = sku.group(1).strip() if sku else "N/A"
                        c_val = category.group(1).strip() if category else "ทั่วไป"

                        if b_val != "ทั่วไป": valid_brands.add(b_val)
                        if c_val != "ทั่วไป": valid_categories.add(c_val)
                        if s_val != "N/A": valid_skus.add(s_val)

                        if "policies" in root or "policy" in filename.lower() or "faq" in filename.lower() or "about" in filename.lower():
                             doc_type = "Policy/FAQ"
                        elif "products" in root:
                             doc_type = "Product"
                        else:
                             doc_type = "Store Info"

                        md_splits = markdown_splitter.split_text(clean_text)

                        for md_split in md_splits:
                            fine_splits = text_splitter.split_text(md_split.page_content)

                            h1 = md_split.metadata.get('H1_Title', 'ไม่ระบุ')
                            h2 = md_split.metadata.get('H2_Section', 'รายละเอียดทั่วไป')
                            h3 = md_split.metadata.get('H3_Subsection', '')

                            for fine_text in fine_splits:
                                context_header = f"[บริบท: {doc_type} | ไฟล์: {filename} | หมวด: {c_val} | แบรนด์: {b_val} | รหัส: {s_val} | ราคา: ฿{price_str}]\n"
                                context_header += f"[หัวข้อหลัก: {h1} > {h2}" + (f" > {h3}" if h3 else "") + "]\n"
                                context_header += f"{'-'*30}\n"

                                new_doc = Document(
                                    page_content=context_header + fine_text,
                                    metadata={
                                        "source": filename,
                                        "doc_type": doc_type,
                                        "brand": b_val,
                                        "sku": s_val,
                                        "price_str": price_str,
                                        "min_price": min_price
                                    }
                                )
                                processed_docs.append(new_doc)

                except Exception as e:
                    print(f"❌ Error reading {file_path}: {e}")

    print(f"✅ โหลดเสร็จสิ้น: {file_count} ไฟล์ (สับละเอียดได้ทั้งหมด {len(processed_docs)} Chunks)")

    catalog = {
        "brands": sorted(list(valid_brands)),
        "categories": sorted(list(valid_categories)),
        "skus": sorted(list(valid_skus))
    }

    return processed_docs, catalog

In [15]:
class FahMaiRAG:
    def __init__(self, documents: List[Document], catalog: dict, api_key: str):
        self.catalog = catalog
        self.api_key = api_key

        print("กำลังสร้าง Vector Database และ BM25 Index...")
        self.embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
        self.vector_db = FAISS.from_documents(documents, self.embeddings)
        self.bm25_retriever = BM25Retriever.from_documents(documents)

        print("กำลังโหลด BGE-M3 Reranker...")
        self.reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=2048)
        print("✅ RAG Engine พร้อมลุย!")



    def call_thaillm(self, messages: List[dict], max_tokens: int = 2048, temperature: float = 0.0) -> str:
        url = "http://thaillm.or.th/api/kbtg/v1/chat/completions"
        headers = {
            "Content-Type": "application/json",
            "apikey": self.api_key
        }
        payload = {
            "model": "/model",
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature
        }

        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()

        return response.json()["choices"][0]["message"]["content"].strip()

    def generate_advanced_queries(self, question: str) -> List[str]:
        brands_ref = ", ".join(self.catalog['brands'])

        prompt = f"""หน้าที่ของคุณคือสกัด "คำสำคัญ (Keywords)" จากคำถามของผู้ใช้ เพื่อนำไปค้นหาในฐานข้อมูลคลังสินค้า

[รายการแบรนด์ที่มีในระบบ]: {brands_ref}

กฎเหล็ก:
1. สกัดเฉพาะ "ชื่อรุ่นสินค้า", "ชื่อแบรนด์", "ฟีเจอร์", หรือ "นโยบาย" ที่ปรากฏในคำถามเท่านั้น
2. ตรวจสอบว่าในคำถามมีชื่อแบรนด์จาก [รายการแบรนด์ที่มีในระบบ] หรือไม่ หากมี ให้แยกชื่อแบรนด์ออกมาเป็น 1 คำค้นหาด้วยเสมอ
3. ห้ามคิดเอง ห้ามเติมชื่อแบรนด์หรือตัวเลขราคาที่ "ไม่มีอยู่ในคำถาม" เด็ดขาด
4. ตัดคำสร้อยและคำเชื่อมทิ้ง (เช่น "อยากได้", "มีรุ่นไหนบ้าง", "ครับ", "คะ")
5. ตอบเป็นรายการคำสำคัญ คั่นด้วยเครื่องหมายไปป์ (|) เท่านั้น ห้ามใช้ลูกน้ำ (,) ห้ามมีข้อความอื่น

ตัวอย่างที่ 1:
คำถาม: Watch S3 Ultra ของวงโคจร กันน้ำได้กี่ ATM ครับ
คำค้นหา: Watch S3 Ultra | วงโคจร | กันน้ำ | ATM

ตัวอย่างที่ 2:
คำถาม: ซื้อ ArcWave SoundPillar 300 มา ทำตกพื้นเสียหาย จะเคลมประกันหรือ Care+ ได้ไหมครับ
คำค้นหา: ArcWave | SoundPillar 300 | ตกพื้น | เคลมประกัน | Care+

คำถาม: {question}
คำค้นหา:"""

        try:
            messages = [{"role": "user", "content": prompt}]

            expanded = self.call_thaillm(messages, max_tokens=4096, temperature=0.0)

            expanded = re.sub(r'<think>.*?</think>', '', expanded, flags=re.DOTALL).strip()

            extracted_queries = [q.strip() for q in expanded.split('|') if len(q.strip()) > 2]

            final_queries = []
            final_queries.append(question)

            for q in extracted_queries:
                if q.lower() not in question.lower() or q != question:
                    if q not in final_queries:
                        final_queries.append(q)

            return final_queries[:4]
        except Exception as e:
            print(f"⚠️ API Error (Query Extraction): {e}")
            return [question]

    def extract_budget(self, question: str):
        match = re.search(r'(?:งบ|ไม่เกิน|ต่ำกว่า|มีเงิน|ราคา|ราคาไม่เกิน)\s*฿?([\d,]+)', question)
        if match:
            return int(match.group(1).replace(',', ''))
        return None

    def retrieve(self, question: str):
        queries = self.generate_advanced_queries(question)
        budget = self.extract_budget(question)
        print(f"🔍 Queries: {queries} | งบประมาณ: {budget}")

        final_docs = []

        policy_keywords = [
            "ประกัน", "เคลม", "ซ่อม", "พัง", "แตก", "ตกพื้น", "เสียหาย",
            "care+", "on-site", "คุ้มครอง", "รับที่บ้าน", "ค่าซ่อม",
            "จัดส่ง", "ส่งของ", "ส่งด่วน", "express", "ส่งไปเกาะ",
            "วันทำการ", "ค่าส่ง", "ชั้น", "ลิฟต์", "สถานะ",
            "คืน", "ยกเลิก", "เปลี่ยนใจ", "แกะกล่อง", "แกะใช้แล้ว",
            "clearance", "ลดล้างสต็อก", "mega sale", "pre-order", "พรีออเดอร์", "กี่ชิ้น",
            "จ่าย", "ชำระเงิน", "crypto", "bitcoin", "เทิร์น",
            "trade-in", "เครื่องเก่า", "เสียค่า", "ค่าดำเนินการ",
            "สมาชิก", "platinum", "gold", "points", "พอยต์", "แต้ม"
        ]

        is_policy_question = any(kw.lower() in question.lower() for kw in policy_keywords)

        if is_policy_question:
            policy_docs_raw = self.vector_db.similarity_search(question, k=20)
            policy_docs = [d for d in policy_docs_raw if d.metadata.get("doc_type") == "Policy/FAQ"]
            final_docs.extend(policy_docs[:3])

        for q in queries:
            self.bm25_retriever.k = 25
            bm25_res = self.bm25_retriever.invoke(q)
            vec_res = self.vector_db.similarity_search(q, k=25)

            merged_docs = list({d.page_content: d for d in (bm25_res + vec_res)}.values())

            if budget is not None:
                merged_docs = [
                    d for d in merged_docs
                    if d.metadata.get("min_price", 0) <= budget or d.metadata.get("doc_type") == "Policy/FAQ"
                ]

            if not merged_docs:
                continue

            pairs = [[q, d.page_content] for d in merged_docs]
            scores = self.reranker.predict(pairs)
            scored_docs = sorted(zip(merged_docs, scores), key=lambda x: x[1], reverse=True)

            final_docs.extend([d[0] for d in scored_docs[:5]])

        main_vec = self.vector_db.similarity_search(question, k=5)
        final_docs.extend(main_vec)

        unique_docs = list({d.page_content: d for d in final_docs}.values())
        return unique_docs[:8]

    def answer(self, question: str, choices_str: str):
        relevant_docs = self.retrieve(question)
        context = "\n\n".join([d.page_content for d in relevant_docs])

        system_prompt = """คุณคือ AI ผู้ช่วยร้านฟ้าใหม่ หน้าที่ของคุณคือตอบข้อสอบแบบปรนัยโดยอ้างอิงจาก <context> ที่กำหนดให้เท่านั้น

กฎเหล็กสำหรับการตอบ:
1. คุณต้องแสดงวิธีคิด วิเคราะห์ และคำนวณในแท็ก <reasoning> ... </reasoning> ก่อนเสมอ
2. เมื่อได้ข้อสรุปที่ตรงกับตัวเลือก ให้ตอบเฉพาะ "หมายเลขตัวเลือก" (1-10) ไว้ในแท็ก <answer> ... </answer> เท่านั้น (เช่น <answer>5</answer>) ห้ามพิมพ์ข้อความอื่นในแท็กนี้
3. หากใน <context> ไม่มีข้อมูลเพียงพอที่จะยืนยันตัวเลือกใดๆ ให้เลือกตัวเลือก "ไม่มีข้อมูลนี้ในฐานข้อมูล"
4. หากคำถามไม่เกี่ยวกับสินค้าไอที หรือร้านฟ้าใหม่ ให้เลือกตัวเลือก "คำถามนี้ไม่เกี่ยวข้อง..."

สูตรคำนวณและกฎที่ต้องจำให้แม่น (Strict Math):
- กฎการคิด Points: นำราคาสินค้า / 100 -> ปัดเศษทศนิยมทิ้งทันที -> นำมาคูณระดับสมาชิก (Gold x1.5, Platinum x2) -> ปัดเศษทศนิยมทิ้งอีกครั้งเป็นอันจบ!
  (ตัวอย่างที่ถูกต้อง: ราคา 32,990 / 100 = 329.9 ให้ปัดทิ้งเหลือ 329 -> 329 x 1.5 = 493.5 ให้ปัดทิ้งเหลือ 493 Points)
- กฎค่าจัดส่ง: สินค้าหนักเกิน 30kg บวกเพิ่ม 200 บาท, ขึ้นบันไดตั้งแต่ชั้น 4 ขึ้นไป บวกเพิ่มชั้นละ 100 บาท (เช่น ชั้น 6 = นับชั้น 4, 5, 6 รวม 3 ชั้น = 300 บาท)
- นโยบายคืน: สินค้า Mega Sale คืนได้ภายใน 7 วัน, หูฟัง In-ear แกะแล้วไม่รับคืนทุกกรณี, สินค้าลดล้างสต็อก (Clearance) ไม่รับคืนทุกกรณี"""

        user_prompt = f"<context>\n{context}\n</context>\n\nคำถาม: {question}\n\nตัวเลือก:\n{choices_str}"

        try:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]

            ans_raw = self.call_thaillm(messages, max_tokens=8192, temperature=0.0)

            match = re.search(r'<answer>\s*(\d+)\s*</answer>', ans_raw, re.IGNORECASE)

            if match:
                return match.group(1), relevant_docs
            else:
                fallback_match = re.search(r'\b(\d+)\b$', ans_raw)
                return fallback_match.group(1) if fallback_match else "Error", relevant_docs

        except Exception as e:
            print(f"⚠️ API Error (Answering): {e}")
            return "Error", relevant_docs

In [ ]:
if __name__ == "__main__":
    DATA_DIR = "/content/data/knowledge_base"

    MY_API_KEY = "api_key"

    if not os.path.exists(DATA_DIR):
        print(f"⚠️ ไม่พบโฟลเดอร์ {DATA_DIR} กรุณาตรวจสอบ Path")
    else:
        all_docs, cag = load_and_process_docs(DATA_DIR)

        rag_engine = FahMaiRAG(all_docs, cag, api_key=MY_API_KEY)

        df_questions = pd.read_csv("/content/data/questions.csv")

        submission_data = []
        detailed_log_data = []

        print("🚀 เริ่มการรัน RAG Agentic Pipeline (Infinite Retry)...\n" + "="*50)

        for index, row in df_questions.iterrows():
            qid = row['id']
            question = row['question']

            choices_dict = {f"{i}": str(row[f'choice_{i}']).strip() for i in range(1, 11) if pd.notna(row[f'choice_{i}'])}
            choices_str = "\n".join([f"{k}: {v}" for k, v in choices_dict.items()])

            attempt = 1
            while True:
                try:
                    ans, retrieved_docs = rag_engine.answer(question, choices_str)

                    if ans == "Error":
                        raise ValueError("API 502 Bad Gateway หรือตอบกลับมาเป็น Error")

                    answer_text = choices_dict.get(ans, "N/A")

                    print(f"🟢 ID: {qid} | ตอบ: {ans} (สำเร็จในครั้งที่ {attempt})")

                    source_files_list = list(set([doc.metadata.get("source", "Unknown") for doc in retrieved_docs]))

                    detailed_log_data.append({
                        "id": qid,
                        "question": question,
                        "predicted_choice": ans,
                        "answer_text": answer_text,
                        "source_files": ", ".join(source_files_list)
                    })
                    submission_data.append({"id": qid, "answer": ans})

                    break

                except Exception as e:
                    # ❌ ถ้า Error ให้ปริ้นท์บอก และวนลูปใหม่
                    print(f"⚠️ พลาดครั้งที่ {attempt} (ID {qid}): {e}")
                    print("⏳ รอ 3 วินาทีแล้วยิงใหม่ (Retry จนกว่าจะสำเร็จ)...")
                    time.sleep(3)
                    attempt += 1

        # บันทึกไฟล์เมื่อครบ 100 ข้อ
        pd.DataFrame(submission_data).to_csv("submission_kbtg_final.csv", index=False)
        pd.DataFrame(detailed_log_data).to_csv("detailed_log_kbtg_final.csv", index=False, encoding='utf-8-sig')
        print("✅ รันเสร็จสมบูรณ์ 100 ข้อ!")

กำลังสกัด Metadata และสร้าง Deep Chunks...
✅ โหลดเสร็จสิ้น: 118 ไฟล์ (สับละเอียดได้ทั้งหมด 869 Chunks)
กำลังสร้าง Vector Database และ BM25 Index...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

กำลังโหลด BGE-M3 Reranker...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ RAG Engine พร้อมลุย!
🚀 เริ่มการรัน RAG Agentic Pipeline (Infinite Retry)...
🔍 Queries: ['Watch S3 Ultra กันน้ำได้กี่ ATM ครับ', 'Watch S3 Ultra', 'กันน้ำ', 'ATM'] | งบประมาณ: None
🟢 ID: 1 | ตอบ: 5 (สำเร็จในครั้งที่ 1)
🔍 Queries: ['ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ', 'SaiFah', 'Pen Gen 2', 'ราคา'] | งบประมาณ: None
🟢 ID: 2 | ตอบ: 7 (สำเร็จในครั้งที่ 1)
🔍 Queries: ['หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ', 'HeadPro X1', 'Bluetooth', 'เวอร์ชัน'] | งบประมาณ: None
⚠️ API Error (Answering): 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/kbtg/v1/chat/completions
⚠️ พลาดครั้งที่ 1 (ID 3): API 502 Bad Gateway หรือตอบกลับมาเป็น Error
⏳ รอ 3 วินาทีแล้วยิงใหม่ (Retry จนกว่าจะสำเร็จ)...
🔍 Queries: ['หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ', 'HeadPro X1', 'Bluetooth', 'เวอร์ชัน'] | งบประมาณ: None
🟢 ID: 3 | ตอบ: 2 (สำเร็จในครั้งที่ 2)
🔍 Queries: ['อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเครื่องใหม่ที่ฟ้าใหม่ได้ไหม?', 'SaiFah', 'เทิร์น', 'เครื่องใหม่'] | งบประมาณ

In [19]:
from google.colab import files
files.download('submission_kbtg_final.csv')
files.download('detailed_log_kbtg_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>